# Problem 29: Async Tool with Polling

**Difficulty:** Hard | **Framework:** CrewAI

**Categories:** tool-calling, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/async-tool-with-polling).

## 1. Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must not block indefinitely waiting for the job to complete
- The agent must poll the job status at reasonable intervals
- The agent must stop polling after the job completes or a timeout is reached
- The final result must be fetched and presented to the user


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai import LLM
from crewai.tools import tool
import time

llm = LLM(model="gpt-4o-mini")
job_store = {}

@tool
def generate_report(topic: str) -> str:
    """Start a long-running report generation job."""
    # BUG: This either blocks forever (bad) or returns immediately without results (also bad)
    # TODO: Split into start_job, check_status, and get_result tools
    job_id = f"JOB-{hash(topic) % 10000:04d}"
    job_store[job_id] = {"status": "processing", "topic": topic, "started": time.time()}
    time.sleep(3)
    job_store[job_id]["status"] = "completed"
    job_store[job_id]["result"] = f"Comprehensive report on {topic}: Key findings include market growth of 25%, emerging trends in AI adoption, and regulatory changes expected in Q3."
    return job_store[job_id]["result"]

agent = Agent(
    role="Report Generator",
    goal="Generate detailed reports on topics",
    backstory="You are an assistant that generates detailed reports on topics.",
    tools=[generate_report],
    llm=llm,
)

task = Task(description="Generate a detailed report on the electric vehicle market", expected_output="A report", agent=agent)
crew = Crew(agents=[agent], tasks=[task], process=Process.sequential)
result = crew.kickoff()
print(result)

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Split the workflow into three tools: start_job (returns job ID), check_status (returns status), and get_result (returns final output)
2. In LangGraph, use a loop node that calls check_status and conditionally exits when the job is complete
3. Add a maximum retry count or timeout to prevent infinite polling


## 7. Evaluation Criteria

Check your solution against these criteria:

- Job starts without blocking
- Polling is implemented
- Timeout prevents infinite loops
- Final result is retrieved
